# 26 — Targeted depth search v3: private and dense families
Runs the private gate/private branch 2×2 interaction, the head interaction, adapter rank/depth combinations, pooled and matched-dense controls, and deterministic seed promotion.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDY_CONFIG = 'config/targeted_depth_search_v3.yaml'
PREFIX = 'nfv3_4way_depth_v3'
PHASE = '2'
EXECUTE = False
MAX_NEW_RUNS = 1
RUN_TESTS = True
# ==================================================================

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO
url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)


In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_targeted_depth_search.py'], check=True)
command = [sys.executable, '-u', '-m', 'training.depth_search', '--study-config', STUDY_CONFIG, '--phase', PHASE, '--prefix', PREFIX]
subprocess.run(command, check=True, env=os.environ.copy())
if EXECUTE:
    launch = [*command, '--execute']
    if MAX_NEW_RUNS is not None: launch += ['--max-new-runs', str(MAX_NEW_RUNS)]
    subprocess.run(launch, check=True, env=os.environ.copy())
state = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{PREFIX}_summary' / 'study_state.json'
print(json.dumps(json.loads(state.read_text()), indent=2) if state.is_file() else f'No state yet: {state}')